# 230968078 - Ishan Suryawanshi - Lab9

### Exercise 1: Cybersecurity Threat Detection Agent

In [2]:
from itertools import combinations

def negate(literal):
    if literal.startswith('NOT_'):
        return literal[4:]
    return 'NOT_' + literal

def resolve(ci, cj):
    resolvents = []
    for lit in ci:
        if negate(lit) in cj:
            new_clause = (ci - {lit}) | (cj - {negate(lit)})
            resolvents.append(frozenset(new_clause))
    return resolvents

def resolution_proof(kb_clauses, query_literal, verbose=True):
    neg_query = frozenset({negate(query_literal)})
    clauses   = set(kb_clauses) | {neg_query}

    if verbose:
        print("\n── CNF Clauses ──")
        for i, c in enumerate(sorted(clauses, key=lambda x: sorted(x)), 1):
            print(f"  {i}. {{ {', '.join(sorted(c))} }}")
        print(f"\nNegated query added: {{ {', '.join(sorted(neg_query))} }}")
        print("\n── Resolution Steps ──")

    step = 0
    while True:
        new_clauses = set()
        clause_list = list(clauses)
        for ci, cj in combinations(clause_list, 2):
            resolvents = resolve(ci, cj)
            for r in resolvents:
                if len(r) == 0:          # empty clause → contradiction
                    step += 1
                    if verbose:
                        print(f"  Step {step}: Resolved {{ {', '.join(sorted(ci))} }} "
                              f"with {{ {', '.join(sorted(cj))} }} → {{ }} (empty clause)")
                    return True
                if r not in clauses:
                    step += 1
                    if verbose:
                        print(f"  Step {step}: Resolved {{ {', '.join(sorted(ci))} }} "
                              f"with {{ {', '.join(sorted(cj))} }} → {{ {', '.join(sorted(r))} }}")
                    new_clauses.add(r)

        if not new_clauses:
            return False   # no new clauses → cannot prove
        clauses |= new_clauses


RULES_CYBER = [
    frozenset({'NOT_MultipleFailedLogins', 'NOT_LoginFromUnknownIP', 'SuspiciousLogin'}),
    frozenset({'NOT_SuspiciousLogin', 'NOT_AdminPrivileges', 'HighRiskIntrusion'}),
    frozenset({'NOT_HighRiskIntrusion', 'IntrusionDetected'}),
    frozenset({'NOT_AccessToSensitiveFiles', 'AdminPrivileges', 'IntrusionDetected'}),
]

def run_cyber_scenario(scenario_name, facts, expected):
    print("=" * 60)
    print(f"SCENARIO: {scenario_name}")
    print(f"Facts   : {facts}")
    print(f"Expected: IntrusionDetected = {expected}")
    fact_clauses = [frozenset({f}) for f in facts]
    kb = RULES_CYBER + fact_clauses
    result = resolution_proof(kb, 'IntrusionDetected', verbose=True)
    print(f"\n>>> IntrusionDetected = {result}")
    print("=" * 60)

In [3]:
# ── Scenario 1: Low Risk ──────────────────────────────────────
run_cyber_scenario(
    scenario_name = "Scenario 1 – Low Risk",
    facts         = ['MultipleFailedLogins'],
    expected      = False
)

SCENARIO: Scenario 1 – Low Risk
Facts   : ['MultipleFailedLogins']
Expected: IntrusionDetected = False

── CNF Clauses ──
  1. { AdminPrivileges, IntrusionDetected, NOT_AccessToSensitiveFiles }
  2. { HighRiskIntrusion, NOT_AdminPrivileges, NOT_SuspiciousLogin }
  3. { IntrusionDetected, NOT_HighRiskIntrusion }
  4. { MultipleFailedLogins }
  5. { NOT_IntrusionDetected }
  6. { NOT_LoginFromUnknownIP, NOT_MultipleFailedLogins, SuspiciousLogin }

Negated query added: { NOT_IntrusionDetected }

── Resolution Steps ──
  Step 1: Resolved { NOT_IntrusionDetected } with { AdminPrivileges, IntrusionDetected, NOT_AccessToSensitiveFiles } → { AdminPrivileges, NOT_AccessToSensitiveFiles }
  Step 2: Resolved { NOT_IntrusionDetected } with { IntrusionDetected, NOT_HighRiskIntrusion } → { NOT_HighRiskIntrusion }
  Step 3: Resolved { NOT_LoginFromUnknownIP, NOT_MultipleFailedLogins, SuspiciousLogin } with { HighRiskIntrusion, NOT_AdminPrivileges, NOT_SuspiciousLogin } → { HighRiskIntrusion, NOT_Admi

In [4]:
# ── Scenario 2: Suspicious Activity ──────────────────────────
run_cyber_scenario(
    scenario_name = "Scenario 2 – Suspicious Activity",
    facts         = ['MultipleFailedLogins', 'LoginFromUnknownIP'],
    expected      = False
)

SCENARIO: Scenario 2 – Suspicious Activity
Facts   : ['MultipleFailedLogins', 'LoginFromUnknownIP']
Expected: IntrusionDetected = False

── CNF Clauses ──
  1. { AdminPrivileges, IntrusionDetected, NOT_AccessToSensitiveFiles }
  2. { HighRiskIntrusion, NOT_AdminPrivileges, NOT_SuspiciousLogin }
  3. { IntrusionDetected, NOT_HighRiskIntrusion }
  4. { LoginFromUnknownIP }
  5. { MultipleFailedLogins }
  6. { NOT_IntrusionDetected }
  7. { NOT_LoginFromUnknownIP, NOT_MultipleFailedLogins, SuspiciousLogin }

Negated query added: { NOT_IntrusionDetected }

── Resolution Steps ──
  Step 1: Resolved { NOT_IntrusionDetected } with { AdminPrivileges, IntrusionDetected, NOT_AccessToSensitiveFiles } → { AdminPrivileges, NOT_AccessToSensitiveFiles }
  Step 2: Resolved { NOT_IntrusionDetected } with { IntrusionDetected, NOT_HighRiskIntrusion } → { NOT_HighRiskIntrusion }
  Step 3: Resolved { LoginFromUnknownIP } with { NOT_LoginFromUnknownIP, NOT_MultipleFailedLogins, SuspiciousLogin } → { NOT_Mul

In [5]:
# ── Scenario 3: High Risk Intrusion ──────────────────────────
run_cyber_scenario(
    scenario_name = "Scenario 3 – High Risk Intrusion",
    facts         = ['MultipleFailedLogins', 'LoginFromUnknownIP', 'AdminPrivileges'],
    expected      = True
)

SCENARIO: Scenario 3 – High Risk Intrusion
Facts   : ['MultipleFailedLogins', 'LoginFromUnknownIP', 'AdminPrivileges']
Expected: IntrusionDetected = True

── CNF Clauses ──
  1. { AdminPrivileges }
  2. { AdminPrivileges, IntrusionDetected, NOT_AccessToSensitiveFiles }
  3. { HighRiskIntrusion, NOT_AdminPrivileges, NOT_SuspiciousLogin }
  4. { IntrusionDetected, NOT_HighRiskIntrusion }
  5. { LoginFromUnknownIP }
  6. { MultipleFailedLogins }
  7. { NOT_IntrusionDetected }
  8. { NOT_LoginFromUnknownIP, NOT_MultipleFailedLogins, SuspiciousLogin }

Negated query added: { NOT_IntrusionDetected }

── Resolution Steps ──
  Step 1: Resolved { AdminPrivileges } with { HighRiskIntrusion, NOT_AdminPrivileges, NOT_SuspiciousLogin } → { HighRiskIntrusion, NOT_SuspiciousLogin }
  Step 2: Resolved { NOT_IntrusionDetected } with { AdminPrivileges, IntrusionDetected, NOT_AccessToSensitiveFiles } → { AdminPrivileges, NOT_AccessToSensitiveFiles }
  Step 3: Resolved { NOT_IntrusionDetected } with { Int

### Wastewater Treatment Decision Agent

In [7]:
RULES_WASTE = [
    frozenset({'NOT_HighBOD', 'NOT_LowDO', 'OrganicPollution'}),
    frozenset({'NOT_HighTurbidity', 'Contamination'}),
    frozenset({'NOT_ToxicChemicals', 'Contamination'}),
    frozenset({'NOT_OrganicPollution', 'NOT_Contamination', 'SeverePollution'}),
    frozenset({'NOT_SeverePollution', 'TreatmentRequired'}),
    frozenset({'NOT_pHImbalance', 'TreatmentRequired'}),
]

def run_waste_scenario(scenario_name, facts, expected):
    print("=" * 60)
    print(f"SCENARIO: {scenario_name}")
    print(f"Facts   : {facts}")
    print(f"Expected: TreatmentRequired = {expected}")
    fact_clauses = [frozenset({f}) for f in facts]
    kb = RULES_WASTE + fact_clauses
    result = resolution_proof(kb, 'TreatmentRequired', verbose=True)
    print(f"\n>>> TreatmentRequired = {result}")
    print("=" * 60)

In [8]:
# ── Scenario 1: Only HighBOD ─────────────────────────────────
run_waste_scenario(
    scenario_name = "Scenario 1 – HighBOD only",
    facts         = ['HighBOD'],
    expected      = False
)

SCENARIO: Scenario 1 – HighBOD only
Facts   : ['HighBOD']
Expected: TreatmentRequired = False

── CNF Clauses ──
  1. { Contamination, NOT_HighTurbidity }
  2. { Contamination, NOT_ToxicChemicals }
  3. { HighBOD }
  4. { NOT_Contamination, NOT_OrganicPollution, SeverePollution }
  5. { NOT_HighBOD, NOT_LowDO, OrganicPollution }
  6. { NOT_SeverePollution, TreatmentRequired }
  7. { NOT_TreatmentRequired }
  8. { NOT_pHImbalance, TreatmentRequired }

Negated query added: { NOT_TreatmentRequired }

── Resolution Steps ──
  Step 1: Resolved { NOT_pHImbalance, TreatmentRequired } with { NOT_TreatmentRequired } → { NOT_pHImbalance }
  Step 2: Resolved { NOT_SeverePollution, TreatmentRequired } with { NOT_Contamination, NOT_OrganicPollution, SeverePollution } → { NOT_Contamination, NOT_OrganicPollution, TreatmentRequired }
  Step 3: Resolved { NOT_SeverePollution, TreatmentRequired } with { NOT_TreatmentRequired } → { NOT_SeverePollution }
  Step 4: Resolved { HighBOD } with { NOT_HighBOD, 

In [9]:
# ── Scenario 2: HighBOD, LowDO, HighTurbidity ────────────────
run_waste_scenario(
    scenario_name = "Scenario 2 – HighBOD + LowDO + HighTurbidity",
    facts         = ['HighBOD', 'LowDO', 'HighTurbidity'],
    expected      = True
)

SCENARIO: Scenario 2 – HighBOD + LowDO + HighTurbidity
Facts   : ['HighBOD', 'LowDO', 'HighTurbidity']
Expected: TreatmentRequired = True

── CNF Clauses ──
  1. { Contamination, NOT_HighTurbidity }
  2. { Contamination, NOT_ToxicChemicals }
  3. { HighBOD }
  4. { HighTurbidity }
  5. { LowDO }
  6. { NOT_Contamination, NOT_OrganicPollution, SeverePollution }
  7. { NOT_HighBOD, NOT_LowDO, OrganicPollution }
  8. { NOT_SeverePollution, TreatmentRequired }
  9. { NOT_TreatmentRequired }
  10. { NOT_pHImbalance, TreatmentRequired }

Negated query added: { NOT_TreatmentRequired }

── Resolution Steps ──
  Step 1: Resolved { Contamination, NOT_ToxicChemicals } with { NOT_Contamination, NOT_OrganicPollution, SeverePollution } → { NOT_OrganicPollution, NOT_ToxicChemicals, SeverePollution }
  Step 2: Resolved { NOT_Contamination, NOT_OrganicPollution, SeverePollution } with { NOT_SeverePollution, TreatmentRequired } → { NOT_Contamination, NOT_OrganicPollution, TreatmentRequired }
  Step 3: R

In [10]:
# ── Scenario 3: ToxicChemicals only ──────────────────────────
run_waste_scenario(
    scenario_name = "Scenario 3 – ToxicChemicals only",
    facts         = ['ToxicChemicals'],
    expected      = True   # Contamination alone is NOT enough; but let's verify
)
print("\n--- Extended: ToxicChemicals + pHImbalance ---")
run_waste_scenario(
    scenario_name = "Scenario 3 Extended – ToxicChemicals + pHImbalance",
    facts         = ['ToxicChemicals', 'pHImbalance'],
    expected      = True
)

SCENARIO: Scenario 3 – ToxicChemicals only
Facts   : ['ToxicChemicals']
Expected: TreatmentRequired = True

── CNF Clauses ──
  1. { Contamination, NOT_HighTurbidity }
  2. { Contamination, NOT_ToxicChemicals }
  3. { NOT_Contamination, NOT_OrganicPollution, SeverePollution }
  4. { NOT_HighBOD, NOT_LowDO, OrganicPollution }
  5. { NOT_SeverePollution, TreatmentRequired }
  6. { NOT_TreatmentRequired }
  7. { NOT_pHImbalance, TreatmentRequired }
  8. { ToxicChemicals }

Negated query added: { NOT_TreatmentRequired }

── Resolution Steps ──
  Step 1: Resolved { NOT_pHImbalance, TreatmentRequired } with { NOT_TreatmentRequired } → { NOT_pHImbalance }
  Step 2: Resolved { ToxicChemicals } with { Contamination, NOT_ToxicChemicals } → { Contamination }
  Step 3: Resolved { NOT_SeverePollution, TreatmentRequired } with { NOT_Contamination, NOT_OrganicPollution, SeverePollution } → { NOT_Contamination, NOT_OrganicPollution, TreatmentRequired }
  Step 4: Resolved { NOT_SeverePollution, Treatme

In [11]:
# ── Summary Table ─────────────────────────────────────────────
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

# Problem 1
cyber_tests = [
    ("P1-S1", "Cybersecurity", ['MultipleFailedLogins'], 'IntrusionDetected', False),
    ("P1-S2", "Cybersecurity", ['MultipleFailedLogins','LoginFromUnknownIP'], 'IntrusionDetected', False),
    ("P1-S3", "Cybersecurity", ['MultipleFailedLogins','LoginFromUnknownIP','AdminPrivileges'], 'IntrusionDetected', True),
]
waste_tests = [
    ("P2-S1", "Wastewater", ['HighBOD'], 'TreatmentRequired', False),
    ("P2-S2", "Wastewater", ['HighBOD','LowDO','HighTurbidity'], 'TreatmentRequired', True),
    ("P2-S3", "Wastewater", ['ToxicChemicals','pHImbalance'], 'TreatmentRequired', True),
]

print(f"{'ID':<8} {'Domain':<15} {'Query':<22} {'Expected':<10} {'Got':<10} {'Pass?'}")
print("-"*70)

for tid, domain, facts, query, expected in cyber_tests + waste_tests:
    rules = RULES_CYBER if domain == "Cybersecurity" else RULES_WASTE
    kb = rules + [frozenset({f}) for f in facts]
    got = resolution_proof(kb, query, verbose=False)
    status = "✓ PASS" if got == expected else "✗ FAIL"
    print(f"{tid:<8} {domain:<15} {query:<22} {str(expected):<10} {str(got):<10} {status}")



SUMMARY
ID       Domain          Query                  Expected   Got        Pass?
----------------------------------------------------------------------
P1-S1    Cybersecurity   IntrusionDetected      False      False      ✓ PASS
P1-S2    Cybersecurity   IntrusionDetected      False      False      ✓ PASS
P1-S3    Cybersecurity   IntrusionDetected      True       True       ✓ PASS
P2-S1    Wastewater      TreatmentRequired      False      False      ✓ PASS
P2-S2    Wastewater      TreatmentRequired      True       True       ✓ PASS
P2-S3    Wastewater      TreatmentRequired      True       True       ✓ PASS
